In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 設定（ここを変えてA/Bテスト）
# ============================================================
MIXUP_N_2WAY     = 2000    # 2-way mixup生成数 (元: 500)
MIXUP_N_3WAY     = 1000    # 3-way mixup生成数 (新規)
MIXUP_ALPHA_2WAY = 0.3     # Beta分布α (試す値: 0.2, 0.3, 0.5, 1.0)
MIXUP_ALPHA_3WAY = 0.5     # Dirichlet α
N_ENSEMBLE_SEEDS = 5       # Multi-seed数 (1なら従来と同じ)
AUG_SAMPLE_WEIGHT = 0.5    # 拡張データの学習重み (1.0=等重み)

print("=" * 60)
print(f"🧪 Multi-Seed Mixup Ensemble")
print(f"   2-way: {MIXUP_N_2WAY} samples (α={MIXUP_ALPHA_2WAY})")
print(f"   3-way: {MIXUP_N_3WAY} samples (α={MIXUP_ALPHA_3WAY})")
print(f"   Seeds: {N_ENSEMBLE_SEEDS}")
print(f"   Aug weight: {AUG_SAMPLE_WEIGHT}")
print("=" * 60)

# ============================================================
# 過去スコア記録
# ============================================================
HISTORY = [
    ("LGB単独(元特徴量)",       17.21, None,  12.615),
    ("元Blend(LGB/PLS/Ridge)", 14.10, None,  12.647),
    ("正則化強化(LGB単独)",     17.60, None,  12.760),
    ("Huber Loss",             17.53, None,  12.770),
    ("PLS予測を特徴量追加",     15.65, None,  12.800),
    ("逆距離加重KNN",          17.13, None,  12.940),
    ("d2(二次微分)追加",        16.12, None,  13.410),
    ("物理特徴量53個追加",      12.68, None,  14.500),
    ("Mixup(500, α=0.3)",     None,  None,  11.80),
]

# ============================================================
# 1. データ読み込み
# ============================================================
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

wavenumbers = np.array([float(c) for c in spec_cols])
wavelengths = np.where(wavenumbers > 0, 10000000 / wavenumbers, 0)
idx_1940 = np.argmin(np.abs(wavelengths - 1940))
idx_1300 = np.argmin(np.abs(wavelengths - 1300))


def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s


# ============================================================
# 2. Augmentation関数
# ============================================================
def mixup_2way(X, y, species, n_augment, alpha, rng):
    """異なる2樹種のスペクトルを線形補間"""
    unique_sp = np.unique(species)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sp1, sp2 = rng.choice(unique_sp, size=2, replace=False)
        i1 = rng.choice(np.where(species == sp1)[0])
        i2 = rng.choice(np.where(species == sp2)[0])
        lam = rng.beta(alpha, alpha)
        X_aug.append(lam * X[i1] + (1 - lam) * X[i2])
        y_aug.append(lam * y[i1] + (1 - lam) * y[i2])
    return np.array(X_aug), np.array(y_aug)


def mixup_3way(X, y, species, n_augment, alpha, rng):
    """異なる3樹種のスペクトルをDirichlet重みで混合"""
    unique_sp = np.unique(species)
    if len(unique_sp) < 3:
        return np.empty((0, X.shape[1])), np.empty(0)
    X_aug, y_aug = [], []
    for _ in range(n_augment):
        sps = rng.choice(unique_sp, size=3, replace=False)
        idxs = [rng.choice(np.where(species == sp)[0]) for sp in sps]
        weights = rng.dirichlet([alpha] * 3)
        x_mix = sum(w * X[i] for w, i in zip(weights, idxs))
        y_mix = sum(w * y[i] for w, i in zip(weights, idxs))
        X_aug.append(x_mix)
        y_aug.append(y_mix)
    return np.array(X_aug), np.array(y_aug)


# ============================================================
# 3. 特徴量作成
# ============================================================
def make_features(X_raw):
    snv = apply_snv(X_raw)
    d1 = savgol_filter(snv, window_length=15, polyorder=2, deriv=1, axis=1)
    ratio = (X_raw[:, idx_1940] / (X_raw[:, idx_1300] + 1e-8)).reshape(-1, 1)
    std = np.std(X_raw, axis=1, keepdims=True)
    return snv, d1, ratio, std


# ============================================================
# 4. Multi-Seed Ensemble CV
# ============================================================
gkf = GroupKFold(n_splits=5)
X_test_raw = test[spec_cols].values

all_seed_test = []
all_seed_oof = []

for seed_idx in range(N_ENSEMBLE_SEEDS):
    base_seed = seed_idx * 1000
    print(f"\n{'='*60}")
    print(f"🌱 Seed {seed_idx+1}/{N_ENSEMBLE_SEEDS} (base={base_seed})")
    print(f"{'='*60}")

    final_lgb = np.zeros(len(test))
    oof_lgb = np.zeros(len(train))
    fold_rmses = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(
        train[spec_cols].values, y_train_log, groups
    )):
        va_species = train.iloc[va_idx]['樹種'].unique()
        tr_sp_nums = groups.iloc[tr_idx].values

        X_tr_raw = train[spec_cols].values[tr_idx]
        y_tr = y_train_log.iloc[tr_idx].values
        X_va_raw = train[spec_cols].values[va_idx]
        y_va = y_train_log.iloc[va_idx].values

        # ── Augmentation ──
        fold_seed = base_seed + fold
        rng = np.random.RandomState(fold_seed)

        X_m2, y_m2 = mixup_2way(X_tr_raw, y_tr, tr_sp_nums,
                                  MIXUP_N_2WAY, MIXUP_ALPHA_2WAY, rng)
        X_m3, y_m3 = mixup_3way(X_tr_raw, y_tr, tr_sp_nums,
                                  MIXUP_N_3WAY, MIXUP_ALPHA_3WAY, rng)

        n_orig = len(X_tr_raw)
        X_tr_aug = np.vstack([X_tr_raw, X_m2, X_m3])
        y_tr_aug = np.concatenate([y_tr, y_m2, y_m3])

        # Sample weights（実データ優先）
        w = np.ones(len(y_tr_aug))
        w[n_orig:] = AUG_SAMPLE_WEIGHT

        if fold == 0 and seed_idx == 0:
            print(f"  元: {n_orig}, 2way: {len(X_m2)}, 3way: {len(X_m3)}")
            print(f"  合計: {len(X_tr_aug)} (aug weight={AUG_SAMPLE_WEIGHT})")

        # ── 特徴量 ──
        snv_tr, d1_tr, ratio_tr, std_tr = make_features(X_tr_aug)
        snv_va, d1_va, ratio_va, std_va = make_features(X_va_raw)
        snv_te, d1_te, ratio_te, std_te = make_features(X_test_raw)

        # PCA（元データのみでfit）
        snv_tr_orig = apply_snv(X_tr_raw)
        pca = PCA(n_components=10, random_state=42)
        pca.fit(snv_tr_orig)
        pca_tr = pca.transform(snv_tr)
        pca_va = pca.transform(snv_va)
        pca_te = pca.transform(snv_te)

        # KNN（元データのみでfit）
        pca_tr_orig = pca.transform(snv_tr_orig)
        knn = NearestNeighbors(n_neighbors=5, metric='cosine')
        knn.fit(pca_tr_orig)

        # KNN特徴量: train
        _, ind_tr = knn.kneighbors(pca_tr, n_neighbors=6)
        knn_ymean_tr = np.zeros(len(X_tr_aug))
        for i in range(len(X_tr_aug)):
            nb = ind_tr[i]
            if i < n_orig:
                valid = nb[nb != i][:5]
            else:
                valid = nb[:5]
            knn_ymean_tr[i] = np.mean(y_tr[valid])
        knn_ymean_tr = knn_ymean_tr.reshape(-1, 1)

        # KNN特徴量: val / test
        _, ind_va = knn.kneighbors(pca_va, n_neighbors=5)
        knn_ymean_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)
        _, ind_te = knn.kneighbors(pca_te, n_neighbors=5)
        knn_ymean_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)

        # 特徴量結合
        feat_tr = np.hstack([snv_tr, d1_tr, pca_tr, knn_ymean_tr, ratio_tr, std_tr])
        feat_va = np.hstack([snv_va, d1_va, pca_va, knn_ymean_va, ratio_va, std_va])
        feat_te = np.hstack([snv_te, d1_te, pca_te, knn_ymean_te, ratio_te, std_te])

        # ── LightGBM ──
        lgb_model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.03,
            max_depth=5, num_leaves=31,
            subsample=0.8, colsample_bytree=0.3,
            min_child_samples=20,
            random_state=42 + seed_idx,
            verbosity=-1
        )
        lgb_model.fit(
            feat_tr, y_tr_aug,
            sample_weight=w,
            eval_set=[(feat_va, y_va)],
            callbacks=[lgb.early_stopping(30, verbose=False)]
        )

        p_va = np.expm1(lgb_model.predict(feat_va))
        p_te = np.expm1(lgb_model.predict(feat_te))
        oof_lgb[va_idx] = p_va
        final_lgb += p_te / 5

        rmse = np.sqrt(mean_squared_error(np.expm1(y_va), p_va))
        fold_rmses.append(rmse)
        print(f"  Fold {fold+1} RMSE: {rmse:.4f}  (valid species: {list(va_species)})")

    oof_rmse = np.sqrt(mean_squared_error(np.expm1(y_train_log), oof_lgb))
    fold_mean = np.mean(fold_rmses)
    fold_std = np.std(fold_rmses)
    print(f"  🌟 Seed {seed_idx+1} OOF RMSE: {oof_rmse:.4f} "
          f"(Fold平均: {fold_mean:.4f} ± {fold_std:.4f})")

    all_seed_test.append(final_lgb)
    all_seed_oof.append(oof_lgb)


# ============================================================
# 5. Ensemble集約
# ============================================================
final_ensemble = np.mean(all_seed_test, axis=0)
oof_ensemble = np.mean(all_seed_oof, axis=0)

y_true = np.expm1(y_train_log)
oof_rmse_ens = np.sqrt(mean_squared_error(y_true, oof_ensemble))

print(f"\n{'='*60}")
print(f"📊 Ensemble結果 ({N_ENSEMBLE_SEEDS} seeds)")
print(f"{'='*60}")
print(f"  🌟 Ensemble OOF RMSE: {oof_rmse_ens:.4f}")

# 個別seed間のばらつき
for i, pred in enumerate(all_seed_test):
    diff = np.sqrt(np.mean((pred - final_ensemble)**2))
    print(f"  Seed {i+1} vs Ensemble RMSD: {diff:.4f}")

# 樹種別
print(f"\n  {'樹種':12s} {'n':>4s} {'RMSE':>7s} {'bias':>7s}")
for sp in sorted(train['樹種'].unique()):
    mask = train['樹種'] == sp
    y_s = train.loc[mask, '含水率'].values
    p_s = oof_ensemble[mask.values]
    r = np.sqrt(np.mean((y_s - p_s)**2))
    b = np.mean(y_s - p_s)
    print(f"  {sp:12s} {len(y_s):4d} {r:7.2f} {b:+7.2f}")


# ============================================================
# 6. 提出
# ============================================================
final_out = np.clip(final_ensemble, 0, None)
submit[1] = final_out
fname = (f'submission_multiseed{N_ENSEMBLE_SEEDS}'
         f'_2way{MIXUP_N_2WAY}_3way{MIXUP_N_3WAY}.csv')
submit.to_csv(fname, index=False, header=False)

print(f"\n✅ 提出ファイル: {fname}")
print(f"📈 予測分布: min={final_out.min():.1f}%, "
      f"median={np.median(final_out):.1f}%, "
      f"max={final_out.max():.1f}%")


# ============================================================
# 7. 過去スコア一覧 + 今回の結果
# ============================================================
print(f"\n{'='*60}")
print("📌 全スコア比較（過去 → 今回）")
print(f"{'='*60}")
print(f"  {'手法':<30s} {'OOF RMSE':>10s} {'Fold平均':>10s} {'LB RMSE':>10s}")
print(f"  {'─'*30} {'─'*10} {'─'*10} {'─'*10}")

for name, oof, fold_avg, lb in HISTORY:
    oof_str = f"{oof:.2f}" if oof is not None else "---"
    fold_str = f"{fold_avg:.2f}" if fold_avg is not None else "---"
    lb_str = f"{lb:.3f}" if lb is not None else "---"
    marker = " ← 前BEST" if lb is not None and lb == 11.80 else ""
    print(f"  {name:<30s} {oof_str:>10s} {fold_str:>10s} {lb_str:>10s}{marker}")

# 今回のスコアを表示
today_name = (f"MultiSeed{N_ENSEMBLE_SEEDS}_2w{MIXUP_N_2WAY}_3w{MIXUP_N_3WAY}")
print(f"  {today_name:<30s} {oof_rmse_ens:>10.2f} {'---':>10s} {'???':>10s} ← 今回")

print(f"\n{'='*60}")
print("📊 次のアクション候補")
print(f"{'='*60}")
print("""
  LB改善した場合:
    → α値を変更 (2way: 0.2/0.5, 3way: 0.3/1.0)
    → 生成数を変更 (2way: 3000, 3way: 2000)
    → AUG_SAMPLE_WEIGHT調整 (0.3, 0.7, 1.0)
    
  LB悪化した場合:
    → 3wayを除外 (2wayのみに戻す)
    → 生成数を減らす (2way: 500, 3way: 0)
    → Multi-seed数を減らす (3 seeds)
    → α値を小さく (0.1) → 元データに近い混合
""")

🧪 Multi-Seed Mixup Ensemble
   2-way: 2000 samples (α=0.3)
   3-way: 1000 samples (α=0.5)
   Seeds: 5
   Aug weight: 0.5

🌱 Seed 1/5 (base=0)
  元: 940, 2way: 2000, 3way: 1000
  合計: 3940 (aug weight=0.5)
  Fold 1 RMSE: 6.6720  (valid species: ['ウエンジ', 'トチ'])
  Fold 2 RMSE: 15.9709  (valid species: ['チェリー', 'ヒノキ'])
  Fold 3 RMSE: 18.9156  (valid species: ['ウォールナット', 'クリ'])
  Fold 4 RMSE: 20.6716  (valid species: ['ナラ', 'ベイマツ', 'ホワイトオーク'])
  Fold 5 RMSE: 23.6637  (valid species: ['イチョウ', 'スプルース', '米ヒバ'])
  🌟 Seed 1 OOF RMSE: 18.0595 (Fold平均: 17.1788 ± 5.8159)

🌱 Seed 2/5 (base=1000)
  Fold 1 RMSE: 6.8469  (valid species: ['ウエンジ', 'トチ'])
  Fold 2 RMSE: 16.0115  (valid species: ['チェリー', 'ヒノキ'])
  Fold 3 RMSE: 18.7782  (valid species: ['ウォールナット', 'クリ'])
  Fold 4 RMSE: 21.4998  (valid species: ['ナラ', 'ベイマツ', 'ホワイトオーク'])
  Fold 5 RMSE: 25.0040  (valid species: ['イチョウ', 'スプルース', '米ヒバ'])
  🌟 Seed 2 OOF RMSE: 18.6351 (Fold平均: 17.6281 ± 6.1573)

🌱 Seed 3/5 (base=2000)
  Fold 1 RMSE: 7.0473  (valid